# 🎮 Sistem Rekomendasi Game Rental PlayStation
## Item-Based Collaborative Filtering

**Dataset:**
- **Training** : `Dataset_Game_RAWG__1_.csv` — 2000 game dari katalog global RAWG
- **Testing**  : `dataset_list_game_dan_playtime_rad_playstation.xlsx` — 28 game stok rental PS

**Alur Notebook:**
1. 📦 Load Data
2. 🔧 Preprocessing
3. 🤖 Build Model (Cosine Similarity)
4. 🎯 Fungsi Rekomendasi
5. 📊 Evaluasi Model

---
> Pastikan kedua file data berada di **folder yang sama** dengan notebook ini.


## 📦 Instalasi & Import Library

In [1]:
# Install dependensi (jalankan sekali jika belum terpasang)
# !pip install pandas numpy scikit-learn openpyxl

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from collections import Counter
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

print("✅ Semua library berhasil diimport")
print(f"   pandas     : {pd.__version__}")
print(f"   numpy      : {np.__version__}")


✅ Semua library berhasil diimport
   pandas     : 3.0.2
   numpy      : 2.4.4


## ⚙️ Konfigurasi Path Data

In [2]:
# ── Sesuaikan path jika file berada di lokasi berbeda ──
TRAIN_PATH = "Dataset_Game_RAWG_(1).csv"
TEST_PATH  = "dataset_list_game_dan_playtime_rad_playstation.xlsx"

TOP_K               = 10     # Jumlah rekomendasi yang dikembalikan
SIMILARITY_THRESHOLD = 0.25  # Minimal cosine similarity agar dipertimbangkan


---
## 📂 Bagian 1 — Load Data
Membaca data training (RAWG CSV, separator=`;`) dan data testing (stok rental XLSX).


In [3]:
# ── 1.1 Load Training Data (RAWG)
df_train_raw = pd.read_csv(TRAIN_PATH, sep=";")

print("=" * 55)
print("  DATA TRAINING — RAWG Dataset")
print("=" * 55)
print(f"Shape  : {df_train_raw.shape}  (baris × kolom)")
print(f"Kolom  : {df_train_raw.columns.tolist()}")


  DATA TRAINING — RAWG Dataset
Shape  : (2000, 14)  (baris × kolom)
Kolom  : ['Game_ID', 'Judul', 'Tanggal_Rilis', 'Bisa_PS4', 'Bisa_PS5', 'Rating_Global', 'Jumlah_Rating', 'Metacritic_Score', 'Jumlah_Wishlist', 'Waktu_Main_Jam', 'Batas_Usia', 'Local_Multiplayer', 'Genre', 'Size_GB']


In [4]:
# ── 1.2 Preview Training Data
df_train_raw.head(5)


,Game_ID,Judul,Tanggal_Rilis,Bisa_PS4,Bisa_PS5,Rating_Global,Jumlah_Rating,Metacritic_Score,Jumlah_Wishlist,Waktu_Main_Jam,Batas_Usia,Local_Multiplayer,Genre,Size_GB
0,1007653,EA SPORTS FC 26,25/09/2025,No,Yes,3.08,13,NaN,36,11,Not Rated,Yes,"Simulation, Sports",43.97
1,989185,EA SPORTS FC 25,26/09/2024,No,Yes,3.29,40,NaN,98,41,Not Rated,Yes,"Simulation, Sports",39.70
2,545029,eFootball 2022,30/09/2021,Yes,Yes,2.12,87,30.0,642,1,Everyone,No,Sports,48.54
3,1010089,NBA 2K26,04/09/2025,No,Yes,0.00,0,NaN,5,3,Not Rated,Yes,Sports,68.57
4,850705,Tekken 8,25/01/2024,No,Yes,4.03,69,NaN,387,19,Not Rated,Yes,"Action, Fighting",79.18


In [5]:
# ── 1.3 Informasi tipe data & null values Training
print("Tipe Data & Nilai Kosong — Training:")
info = pd.DataFrame({
    "Tipe"      : df_train_raw.dtypes,
    "Null"      : df_train_raw.isnull().sum(),
    "Null (%)"  : (df_train_raw.isnull().sum() / len(df_train_raw) * 100).round(1)
})
print(info.to_string())


Tipe Data & Nilai Kosong — Training:
                      Tipe  Null  Null (%)
Game_ID              int64     0       0.0
Judul                  str     0       0.0
Tanggal_Rilis          str     1       0.0
Bisa_PS4               str     0       0.0
Bisa_PS5               str     0       0.0
Rating_Global      float64     0       0.0
Jumlah_Rating        int64     0       0.0
Metacritic_Score   float64   528      26.4
Jumlah_Wishlist      int64     0       0.0
Waktu_Main_Jam       int64     0       0.0
Batas_Usia             str     0       0.0
Local_Multiplayer      str     0       0.0
Genre                  str     5       0.2
Size_GB            float64     0       0.0


In [6]:
# ── 1.4 Load Testing Data (Stok Rental PS)
df_test_raw = pd.read_excel(TEST_PATH)

print("=" * 55)
print("  DATA TESTING — Stok Rental PlayStation")
print("=" * 55)
print(f"Shape  : {df_test_raw.shape}  (baris × kolom)")
print(f"Kolom  : {df_test_raw.columns.tolist()}")


  DATA TESTING — Stok Rental PlayStation
Shape  : (28, 9)  (baris × kolom)
Kolom  : ['Nama_Game', 'Tersedia_Di_Meja', 'Total_Playtime', 'Bisa_PS4', 'Bisa_PS5', 'Frekuensi_Booking', 'Local_Multiplayer', 'Genre', 'Size_GB']


In [7]:
# ── 1.5 Preview Testing Data
df_test_raw


,Nama_Game,Tersedia_Di_Meja,Total_Playtime,Bisa_PS4,Bisa_PS5,Frekuensi_Booking,Local_Multiplayer,Genre,Size_GB
0,EA FC26,1;2;3;4,694,No,Yes,0,Yes,Sports,45
1,E-Football,1;2;3;4,454,Yes,Yes,0,No,Action,50
2,Grand Theft Auto V,3;4,230,Yes,Yes,0,No,Action,95
3,Insomniac Marvel's Spider Man 2,2,403,No,Yes,0,No,Action-Adventure,86
4,Tekken 8,1;2,514,No,Yes,0,Yes,Fighting,80
5,Resident Evil 4 Remake,1,383,No,Yes,0,No,Horror,67
6,Black Desert,1,437,No,Yes,0,No,MMORPG,40
7,Stellar Blade,1,263,No,Yes,0,No,Action,30
8,It Takes Two,1;2;6;7;8,128,Yes,Yes,0,Yes,Party,50
9,UFC 5,2;3,189,No,Yes,0,Yes,Sports,55


In [8]:
# ── 1.6 Statistik Deskriptif Training
print("Statistik Deskriptif — Training (kolom numerik):")
df_train_raw.describe().round(2)


Statistik Deskriptif — Training (kolom numerik):


,Game_ID,Rating_Global,Jumlah_Rating,Metacritic_Score,Jumlah_Wishlist,Waktu_Main_Jam,Size_GB
count,2000.00,2000.00,2000.00,1472.00,2000.00,2000.00,2000.00
mean,134133.11,3.53,399.39,76.22,2362.67,5.23,63.66
std,220113.43,0.60,642.17,8.85,2566.53,10.43,28.96
min,4.00,0.00,0.00,30.00,5.00,0.00,5.05
25%,3334.25,3.17,70.00,71.00,790.50,2.00,44.98
50%,17142.00,3.58,152.00,77.00,1371.00,4.00,64.89
75%,254542.75,3.97,426.50,82.00,2722.25,5.00,87.37
max,1010089.00,4.81,7370.00,97.00,22553.00,329.00,131.67


---
## 🔧 Bagian 2 — Preprocessing

Langkah preprocessing:

**Training (RAWG):**
- Isi `Metacritic_Score` kosong → median
- Isi `Genre` kosong → `'Unknown'`
- Encode `Yes/No` → `1/0`
- Normalisasi `Rating_Global` ke skala 0–100
- Hitung `Popularity_Score` gabungan
- One-Hot Encoding genre (dipisah koma)
- MinMaxScaler semua fitur numerik

**Testing (Rental):**
- Parse `Tersedia_Di_Meja` (`"1;2;3;4"`) → list integer
- Encode `Yes/No` → `1/0`
- Hitung `Playtime_Score` (proxy popularitas)
- One-Hot Encoding genre
- Selaraskan kolom genre dengan training


In [9]:
# ── 2.1 Preprocessing Training Data
df_train = df_train_raw.copy()

# Isi nilai kosong
df_train["Metacritic_Score"] = df_train["Metacritic_Score"].fillna(
    df_train["Metacritic_Score"].median()
)
df_train["Genre"] = df_train["Genre"].fillna("Unknown")
df_train = df_train.dropna(subset=["Tanggal_Rilis"])   # hapus 1 baris

# Encode boolean Yes/No → 1/0
for col in ["Bisa_PS4", "Bisa_PS5", "Local_Multiplayer"]:
    df_train[col] = df_train[col].str.strip().str.lower().map({"yes": 1, "no": 0})

# Normalisasi rating ke 0–100
df_train["Rating_Norm"] = (df_train["Rating_Global"] / 5.0) * 100.0

# Popularity Score gabungan
df_train["Popularity_Score"] = (
    df_train["Jumlah_Rating"]   * 0.30 +
    df_train["Jumlah_Wishlist"] * 0.30 +
    df_train["Rating_Norm"]     * 0.40
)

print("✅ Preprocessing Training selesai")
print(f"   Baris tersisa : {len(df_train)}")
print(f"   Null tersisa  : {df_train[['Metacritic_Score','Genre']].isnull().sum().to_dict()}")


✅ Preprocessing Training selesai
   Baris tersisa : 1999
   Null tersisa  : {'Metacritic_Score': 0, 'Genre': 0}


In [10]:
# ── 2.1b KNN Imputation untuk Genre 'Unknown'
# Menggunakan K-Nearest Neighbors untuk mengisi genre yang hilang
from sklearn.neighbors import KNeighborsClassifier

def impute_unknown_genres(df, feature_cols_for_knn=None):
    """
    Mengisi nilai 'Unknown' di kolom Genre menggunakan KNN classifier.
    Algoritma:
    1. Pisahkan data dengan genre diketahui vs. unknown
    2. Build fitur numerik dari game yang genrenya diketahui
    3. Train KNN classifier
    4. Prediksi genre untuk game dengan genre unknown
    """
    df_work = df.copy()
    
    # Default feature untuk KNN imputation (tanpa genre)
    if feature_cols_for_knn is None:
        feature_cols_for_knn = [
            "Rating_Global", "Jumlah_Rating", "Metacritic_Score",
            "Jumlah_Wishlist", "Waktu_Main_Jam", "Bisa_PS4", "Bisa_PS5",
            "Local_Multiplayer", "Size_GB"
        ]
    
    # Cari baris dengan genre known dan unknown
    mask_known = df_work["Genre"] != "Unknown"
    mask_unknown = df_work["Genre"] == "Unknown"
    
    n_unknown = mask_unknown.sum()
    
    if n_unknown == 0:
        print(f"✅ Tidak ada genre 'Unknown' untuk diisi.")
        return df_work
    
    print(f"🔍 Ditemukan {n_unknown} game dengan genre 'Unknown'")
    print(f"   Menggunakan KNN dengan k=5 untuk imputation...")
    
    # Build feature matrix (hanya untuk game yang fiturnya lengkap)
    X_known = df_work[mask_known][feature_cols_for_knn].fillna(0).values
    y_known = df_work[mask_known]["Genre"].values
    
    X_unknown = df_work[mask_unknown][feature_cols_for_knn].fillna(0).values
    
    # Normalize fitur untuk KNN
    scaler_knn = MinMaxScaler()
    X_known_scaled = scaler_knn.fit_transform(X_known)
    X_unknown_scaled = scaler_knn.transform(X_unknown)
    
    # Train KNN classifier (k=5)
    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X_known_scaled, y_known)
    
    # Prediksi genre untuk game unknown
    y_pred = knn.predict(X_unknown_scaled)
    
    # Update genre di dataframe
    df_work.loc[mask_unknown, "Genre"] = y_pred
    
    print(f"✅ Imputation selesai")
    print(f"   Genre hasil prediksi (first 5):")
    for i, (orig_idx, pred_genre) in enumerate(zip(df_work[mask_unknown].index[:5], y_pred[:5])):
        game_title = df_work.loc[orig_idx, "Judul"]
        print(f"      → {game_title:<50} : {pred_genre}")
    
    return df_work

# Terapkan imputation
df_train = impute_unknown_genres(df_train)
print(f"\n   Genre 'Unknown' tersisa: {(df_train['Genre'] == 'Unknown').sum()}")


🔍 Ditemukan 5 game dengan genre 'Unknown'
   Menggunakan KNN dengan k=5 untuk imputation...
✅ Imputation selesai
   Genre hasil prediksi (first 5):
      → The Playroom VR                                    : Adventure
      → Hardware: Rivals                                   : Casual, Arcade, Simulation
      → EA Play Hub                                        : Casual, Indie
      → Tilt Brush                                         : Adventure
      → PlayStation VR Demo Disc                           : Indie, Adventure

   Genre 'Unknown' tersisa: 0


In [11]:
# ── 2.2 One-Hot Encoding Genre — Training
genre_ohe_train = df_train["Genre"].str.get_dummies(sep=", ")
genre_ohe_train.columns = [
    "genre_" + c.strip().lower().replace(" ", "_").replace("-", "_")
    for c in genre_ohe_train.columns
]
genre_cols = genre_ohe_train.columns.tolist()

df_train = pd.concat(
    [df_train.reset_index(drop=True), genre_ohe_train.reset_index(drop=True)],
    axis=1
)

print(f"Genre unik di katalog RAWG : {df_train_raw['Genre'].nunique()}")
print(f"Kolom genre setelah OHE    : {len(genre_cols)}")
print(f"Genre columns: {genre_cols}")


Genre unik di katalog RAWG : 378
Kolom genre setelah OHE    : 19
Genre columns: ['genre_action', 'genre_adventure', 'genre_arcade', 'genre_board_games', 'genre_card', 'genre_casual', 'genre_educational', 'genre_family', 'genre_fighting', 'genre_indie', 'genre_massively_multiplayer', 'genre_platformer', 'genre_puzzle', 'genre_rpg', 'genre_racing', 'genre_shooter', 'genre_simulation', 'genre_sports', 'genre_strategy']


In [12]:
# ── 2.3 Preprocessing Testing Data
df_test = df_test_raw.copy()

# Parse kolom Tersedia_Di_Meja: "1;2;3;4" → [1,2,3,4]
def parse_meja(val):
    if pd.isna(val):
        return []
    return [int(x.strip()) for x in str(val).split(";") if x.strip().isdigit()]

df_test["Meja_List"]   = df_test["Tersedia_Di_Meja"].apply(parse_meja)
df_test["Jumlah_Meja"] = df_test["Meja_List"].apply(len)

# Encode boolean
for col in ["Bisa_PS4", "Bisa_PS5", "Local_Multiplayer"]:
    df_test[col] = df_test[col].str.strip().str.lower().map({"yes": 1, "no": 0})

# Playtime Score (proxy popularitas)
p_min, p_max = df_test["Total_Playtime"].min(), df_test["Total_Playtime"].max()
df_test["Playtime_Score"] = (
    (df_test["Total_Playtime"] - p_min) / (p_max - p_min) * 100
    if p_max > p_min else 50.0
)

print("✅ Preprocessing Testing selesai")
df_test[["Nama_Game","Genre","Total_Playtime","Meja_List","Jumlah_Meja","Playtime_Score"]]


✅ Preprocessing Testing selesai


,Nama_Game,Genre,Total_Playtime,Meja_List,Jumlah_Meja,Playtime_Score
0,EA FC26,Sports,694,"[1, 2, 3, 4]",4,100.000000
1,E-Football,Action,454,"[1, 2, 3, 4]",4,60.000000
2,Grand Theft Auto V,Action,230,"[3, 4]",2,22.666667
3,Insomniac Marvel's Spider Man 2,Action-Adventure,403,[2],1,51.500000
4,Tekken 8,Fighting,514,"[1, 2]",2,70.000000
5,Resident Evil 4 Remake,Horror,383,[1],1,48.166667
6,Black Desert,MMORPG,437,[1],1,57.166667
7,Stellar Blade,Action,263,[1],1,28.166667
8,It Takes Two,Party,128,"[1, 2, 6, 7, 8]",5,5.666667
9,UFC 5,Sports,189,"[2, 3]",2,15.833333


In [32]:
# ── 2.4 One-Hot Encoding Genre — Testing (selaraskan dengan training)
genre_ohe_test = df_test["Genre"].str.replace("-", " ").str.get_dummies(sep=" ")
genre_ohe_test.columns = [
    "genre_" + c.strip().lower().replace(" ", "_")
    for c in genre_ohe_test.columns
]

# Tambah kolom genre training yang tidak ada di testing → isi 0
for col in genre_cols:
    if col not in genre_ohe_test.columns:
        genre_ohe_test[col] = 0
genre_ohe_test = genre_ohe_test[[c for c in genre_cols if c in genre_ohe_test.columns]]

df_test = pd.concat(
    [df_test.reset_index(drop=True), genre_ohe_test.reset_index(drop=True)],
    axis=1
)

print(f"Genre unik di stok rental : {df_test_raw['Genre'].nunique()}")
print(f"Kolom genre setelah OHE   : {genre_ohe_test.shape[1]}")
genre_ohe_test.head(5)

Genre unik di stok rental : 9
Kolom genre setelah OHE   : 19


,genre_action,genre_adventure,genre_arcade,genre_board_games,genre_card,genre_casual,genre_educational,genre_family,genre_fighting,genre_indie,genre_massively_multiplayer,genre_platformer,genre_puzzle,genre_rpg,genre_racing,genre_shooter,genre_simulation,genre_sports,genre_strategy
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0
1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0


In [14]:
# ── 2.5 Bangun Feature Matrix
NUMERIC_FEATURES = [
    "Rating_Norm", "Metacritic_Score", "Popularity_Score",
    "Waktu_Main_Jam", "Bisa_PS4", "Bisa_PS5", "Local_Multiplayer", "Size_GB",
]
feature_cols = NUMERIC_FEATURES + genre_cols

# ── Feature Matrix Training
X_train = df_train[feature_cols].fillna(0).values.astype(float)

# ── Feature Matrix Testing (kolom tidak tersedia di-proxy atau di-0)
COL_PROXY = {
    "Rating_Norm"      : "Playtime_Score",
    "Metacritic_Score" : "Playtime_Score",
    "Popularity_Score" : "Playtime_Score",
    "Waktu_Main_Jam"   : None,
}

X_test = np.zeros((len(df_test), len(feature_cols)), dtype=float)
for i, col in enumerate(feature_cols):
    if col in df_test.columns:
        X_test[:, i] = df_test[col].fillna(0).values
    elif col in COL_PROXY and COL_PROXY[col] and COL_PROXY[col] in df_test.columns:
        X_test[:, i] = df_test[COL_PROXY[col]].fillna(0).values

# ── MinMaxScaler (fit pada training, transform keduanya)
scaler  = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
for i in range(X_test.shape[1]):
    max_val = scaler.data_max_[i] if scaler.data_max_[i] > 0 else 1
    X_test[:, i] = X_test[:, i] / max_val

print("✅ Feature matrix berhasil dibuat")
print(f"   X_train : {X_train.shape}  (game katalog × fitur)")
print(f"   X_test  : {X_test.shape}  (game stok    × fitur)")
print(f"   Total fitur: {len(feature_cols)}")
print(f"     → {NUMERIC_FEATURES}")
print(f"     → {genre_cols}")


✅ Feature matrix berhasil dibuat
   X_train : (1999, 27)  (game katalog × fitur)
   X_test  : (28, 27)  (game stok    × fitur)
   Total fitur: 27
     → ['Rating_Norm', 'Metacritic_Score', 'Popularity_Score', 'Waktu_Main_Jam', 'Bisa_PS4', 'Bisa_PS5', 'Local_Multiplayer', 'Size_GB']
     → ['genre_action', 'genre_adventure', 'genre_arcade', 'genre_board_games', 'genre_card', 'genre_casual', 'genre_educational', 'genre_family', 'genre_fighting', 'genre_indie', 'genre_massively_multiplayer', 'genre_platformer', 'genre_puzzle', 'genre_rpg', 'genre_racing', 'genre_shooter', 'genre_simulation', 'genre_sports', 'genre_strategy']


---
## 🤖 Bagian 3 — Build Model (Item-Based CF)

**Cara kerja Item-Based Collaborative Filtering:**

Setiap game direpresentasikan sebagai **vektor fitur** (rating, genre, platform, dll).  
Kesamaan antar item dihitung dengan **Cosine Similarity**:

$$\text{sim}(A, B) = \frac{A \cdot B}{\|A\| \times \|B\|}$$

Hasilnya adalah **similarity matrix** berukuran `(28 stok rental × 1999 game RAWG)`.  
Nilai 1.0 = identik, nilai 0.0 = tidak mirip sama sekali.

**Skor akhir rekomendasi:**
$$\text{Skor\_Akhir} = \text{Similarity} \times 0.7 + \text{Popularity} \times 0.3$$


In [15]:
# ── 3.1 Hitung Cosine Similarity Matrix
sim_matrix = cosine_similarity(X_test, X_train)
# Baris = game stok rental (28), Kolom = game RAWG (1999)

print("✅ Similarity matrix berhasil dihitung")
print(f"   Shape  : {sim_matrix.shape}  (stok rental × katalog RAWG)")
print(f"   Min    : {sim_matrix.min():.4f}")
print(f"   Max    : {sim_matrix.max():.4f}")
print(f"   Mean   : {sim_matrix.mean():.4f}")
print(f"   Std    : {sim_matrix.std():.4f}")


✅ Similarity matrix berhasil dihitung
   Shape  : (28, 1999)  (stok rental × katalog RAWG)
   Min    : 0.0120
   Max    : 0.9995
   Mean   : 0.4285
   Std    : 0.1763


In [16]:
# ── 3.2 Preview: Game rental vs. game RAWG paling mirip (top-3 per stok)
print("Top-3 game RAWG paling mirip untuk setiap game stok rental:\n")
for i, row in df_test.iterrows():
    sim_row  = sim_matrix[i]
    top3_idx = np.argsort(sim_row)[::-1][:3]
    matches  = [(df_train.iloc[j]["Judul"], round(float(sim_row[j]), 4)) for j in top3_idx]
    print(f"  {row['Nama_Game']:<42} → {matches}")


Top-3 game RAWG paling mirip untuk setiap game stok rental:

  EA FC26                                    → [('FIFA 22', 0.8807), ('EA SPORTS FC 25', 0.8785), ('NBA 2K26', 0.8776)]
  E-Football                                 → [('Wo Long: Fallen Dynasty', 0.9946), ("Assassin's Creed Mirage", 0.9927), ('Nioh 2', 0.9927)]
  Grand Theft Auto V                         → [('Terminator: Resistance', 0.9564), ("Assassin's Creed Mirage", 0.9482), ('Prince of Persia: The Sands of Time Remake', 0.9462)]
  Insomniac Marvel's Spider Man 2            → [('Forspoken', 0.99), ('Mafia: The Old Country', 0.9894), ('The Medium', 0.9894)]
  Tekken 8                                   → [('Mortal Kombat 1', 0.9045), ('Tekken 8', 0.9037), ('Guilty Gear -Strive', 0.8433)]
  Resident Evil 4 Remake                     → [('Warhammer 40,000: Space Marine II', 0.8385), ('HELLDIVERS 2', 0.8375), ('Marvel Rivals', 0.8371)]
  Black Desert                               → [('Doom: The Dark Ages', 0.832), ('Clair Obs

In [17]:
# ── 4.1 Helper: Filter katalog training
def build_train_mask(ps_version="Both", multiplayer_only=False,
                     genre_filter=None, max_size_gb=None):
    """Buat boolean mask untuk memfilter katalog training."""
    mask = np.ones(len(df_train), dtype=bool)

    if ps_version == "PS4":
        mask &= df_train["Bisa_PS4"].values == 1
    elif ps_version == "PS5":
        mask &= df_train["Bisa_PS5"].values == 1

    if multiplayer_only:
        mask &= df_train["Local_Multiplayer"].values == 1

    if genre_filter:
        gf_lower = [g.lower() for g in genre_filter]
        gmask = df_train["Genre"].str.lower().apply(
            lambda g: any(gf in str(g) for gf in gf_lower)
        ).values
        mask &= gmask

    if max_size_gb is not None:
        mask &= df_train["Size_GB"].values <= max_size_gb

    return mask

print("✅ Fungsi build_train_mask didefinisikan")


✅ Fungsi build_train_mask didefinisikan


In [18]:
# ── 4.2 Fungsi Rekomendasi untuk Pemain
def recommend_player(game_name, top_k=TOP_K, ps_version="PS5",
                     multiplayer_only=False, genre_filter=None,
                     meja=None, max_size_gb=None,
                     similarity_threshold=SIMILARITY_THRESHOLD):
    """
    Rekomendasikan game dari katalog RAWG yang mirip dengan game
    yang dipilih pemain dari stok rental.

    Parameters
    ----------
    game_name         : nama game dari stok rental (contoh: 'Tekken 8')
    top_k             : jumlah rekomendasi yang dikembalikan
    ps_version        : 'PS4' | 'PS5' | 'Both'
    multiplayer_only  : True = hanya game yang bisa multiplayer lokal
    genre_filter      : list genre, contoh ['action', 'fighting']
    meja              : filter game yang tersedia di meja tertentu
    max_size_gb       : batas ukuran maksimal game (GB)
    similarity_threshold : minimal cosine similarity
    """
    # Cari indeks game di stok rental
    names     = df_test["Nama_Game"].str.lower().str.strip()
    mask_exact = names == game_name.lower().strip()
    mask_sub   = names.str.contains(game_name.lower().strip(), na=False)
    test_idx   = (names[mask_exact].index[0] if mask_exact.any() else
                  names[mask_sub].index[0] if mask_sub.any() else None)

    if test_idx is None:
        avail = df_test["Nama_Game"].tolist()
        raise ValueError(f"Game '{game_name}' tidak ditemukan.\nTersedia: {avail}")

    # Validasi meja
    if meja is not None:
        if meja not in df_test.iloc[test_idx]["Meja_List"]:
            raise ValueError(f"'{game_name}' tidak tersedia di Meja {meja}.")

    # Ambil baris similarity untuk game ini
    sim_scores = sim_matrix[test_idx].copy()

    # Terapkan filter
    train_mask = build_train_mask(ps_version, multiplayer_only, genre_filter, max_size_gb)

    # Popularity score ternormalisasi 0-1
    pop_min  = df_train["Popularity_Score"].min()
    pop_max  = df_train["Popularity_Score"].max()
    pop_norm = ((df_train["Popularity_Score"] - pop_min) / (pop_max - pop_min + 1e-9)).values

    # Skor akhir gabungan
    combined = sim_scores * 0.70 + pop_norm * 0.30
    combined[~train_mask] = -1

    top_indices = np.argsort(combined)[::-1]

    results = []
    for idx in top_indices:
        if combined[idx] < 0 or sim_scores[idx] < similarity_threshold:
            continue
        row  = df_train.iloc[idx]
        meta = row["Metacritic_Score"]
        results.append({
            "Rank"          : len(results) + 1,
            "Judul"         : row["Judul"],
            "Genre"         : row["Genre"],
            "Rating_Global" : round(float(row["Rating_Global"]), 2),
            "Metacritic"    : int(meta) if not np.isnan(meta) else "-",
            "Waktu_Main_Jam": int(row["Waktu_Main_Jam"]),
            "Bisa_PS4"      : "Ya" if row["Bisa_PS4"] == 1 else "Tidak",
            "Bisa_PS5"      : "Ya" if row["Bisa_PS5"] == 1 else "Tidak",
            "Multiplayer"   : "Ya" if row["Local_Multiplayer"] == 1 else "Tidak",
            "Size_GB"       : round(float(row["Size_GB"]), 1),
            "Similarity"    : round(float(sim_scores[idx]), 4),
            "Skor_Akhir"    : round(float(combined[idx]), 4),
        })
        if len(results) >= top_k:
            break

    return pd.DataFrame(results)

print("✅ Fungsi recommend_player didefinisikan")


✅ Fungsi recommend_player didefinisikan


In [19]:
# ── 4.3 Fungsi Rekomendasi untuk Pemilik Rental
def recommend_owner(top_k=TOP_K, ps_version="Both",
                    multiplayer_only=False, genre_filter=None):
    """
    Rekomendasikan game dari RAWG yang sebaiknya ditambahkan ke stok rental.
    Berdasarkan rata-rata similarity terhadap keseluruhan stok + popularitas global.
    Game yang sudah ada di stok otomatis dikecualikan.
    """
    avg_sim    = sim_matrix.mean(axis=0)  # rata-rata similarity ke semua stok
    train_mask = build_train_mask(ps_version, multiplayer_only, genre_filter)

    # Kecualikan yang sudah ada di stok
    existing  = set(df_test["Nama_Game"].str.lower().str.strip())
    in_stock  = df_train["Judul"].str.lower().str.strip().isin(existing).values
    train_mask &= ~in_stock

    pop_min  = df_train["Popularity_Score"].min()
    pop_max  = df_train["Popularity_Score"].max()
    pop_norm = ((df_train["Popularity_Score"] - pop_min) / (pop_max - pop_min + 1e-9)).values

    combined = avg_sim * 0.50 + pop_norm * 0.50
    combined[~train_mask] = -1

    top_indices = np.argsort(combined)[::-1]

    results = []
    for idx in top_indices:
        if combined[idx] < 0:
            continue
        row  = df_train.iloc[idx]
        meta = row["Metacritic_Score"]
        results.append({
            "Rank"            : len(results) + 1,
            "Judul"           : row["Judul"],
            "Genre"           : row["Genre"],
            "Rating_Global"   : round(float(row["Rating_Global"]), 2),
            "Metacritic"      : int(meta) if not np.isnan(meta) else "-",
            "Waktu_Main_Jam"  : int(row["Waktu_Main_Jam"]),
            "Bisa_PS4"        : "Ya" if row["Bisa_PS4"] == 1 else "Tidak",
            "Bisa_PS5"        : "Ya" if row["Bisa_PS5"] == 1 else "Tidak",
            "Multiplayer"     : "Ya" if row["Local_Multiplayer"] == 1 else "Tidak",
            "Size_GB"         : round(float(row["Size_GB"]), 1),
            "Popularity_Score": round(float(pop_norm[idx] * 100), 1),
            "Avg_Similarity"  : round(float(avg_sim[idx]), 4),
            "Skor_Akhir"      : round(float(combined[idx]), 4),
        })
        if len(results) >= top_k:
            break

    return pd.DataFrame(results)

print("✅ Fungsi recommend_owner didefinisikan")


✅ Fungsi recommend_owner didefinisikan


In [20]:
# ── 4.4 Demo: Rekomendasi Pemain — Tekken 8 | PS5 | Multiplayer
print("Demo Rekomendasi Pemain: 'Tekken 8' | PS5 | Multiplayer")
reko = recommend_player("Tekken 8", ps_version="PS5", multiplayer_only=True)
reko


Demo Rekomendasi Pemain: 'Tekken 8' | PS5 | Multiplayer


,Rank,Judul,Genre,Rating_Global,Metacritic,Waktu_Main_Jam,Bisa_PS4,Bisa_PS5,Multiplayer,Size_GB,Similarity,Skor_Akhir
0,1,Mortal Kombat 1,"Action, Fighting",3.60,77,10,Tidak,Ya,Ya,95.9,0.9045,0.6398
1,2,Tekken 8,"Action, Fighting",4.03,77,19,Tidak,Ya,Ya,79.2,0.9037,0.6382
2,3,Guilty Gear -Strive,"Action, Fighting",4.24,85,11,Ya,Ya,Ya,109.7,0.8433,0.6016
3,4,Nickelodeon All-Star Brawl,"Action, Fighting",2.86,77,2,Ya,Ya,Ya,48.2,0.8077,0.5787
4,5,Street Fighter 6,"Adventure, Action, Fighting",4.11,77,17,Ya,Ya,Ya,102.4,0.7818,0.5538
5,6,Them's Fightin' Herds,"Indie, Action, Fighting",3.36,82,3,Ya,Ya,Ya,81.8,0.7728,0.5503
6,7,Baldur's Gate III,"Strategy, Adventure, RPG",4.44,97,20,Tidak,Ya,Ya,116.0,0.6749,0.5306
7,8,Sea of Stars,RPG,4.28,90,7,Ya,Ya,Ya,105.0,0.7198,0.5145
8,9,It Takes Two,"Platformer, Adventure, Action",4.46,88,10,Ya,Ya,Ya,97.1,0.6218,0.5097
9,10,Split Fiction,"Adventure, Action",4.45,77,6,Tidak,Ya,Ya,91.6,0.7090,0.5072


In [21]:
# ── 4.5 Demo: Rekomendasi Pemain — Grand Theft Auto V | PS4
print("Demo Rekomendasi Pemain: 'Grand Theft Auto V' | PS4")
reko2 = recommend_player("Grand Theft Auto V", ps_version="PS4")
reko2


Demo Rekomendasi Pemain: 'Grand Theft Auto V' | PS4


,Rank,Judul,Genre,Rating_Global,Metacritic,Waktu_Main_Jam,Bisa_PS4,Bisa_PS5,Multiplayer,Size_GB,Similarity,Skor_Akhir
0,1,Grand Theft Auto V,Action,4.47,92,74,Ya,Ya,Tidak,62.5,0.8083,0.8658
1,2,The Witcher 3: Wild Hunt,"Action, RPG",4.64,92,43,Ya,Ya,Tidak,82.6,0.7634,0.8290
2,3,The Elder Scrolls V: Skyrim,"Action, RPG",4.42,94,46,Ya,Ya,Tidak,72.8,0.7823,0.7662
3,4,Fallout 4,"Action, RPG",3.81,84,38,Ya,Ya,Tidak,99.9,0.8254,0.7556
4,5,Marvel's Spider-Man,Action,4.45,87,6,Ya,Ya,Tidak,49.9,0.8680,0.7506
5,6,Dead by Daylight,Action,3.50,72,10,Ya,Ya,Tidak,82.2,0.9456,0.7328
6,7,Destiny 2,"Shooter, Action",3.52,82,6,Ya,Ya,Tidak,45.5,0.7971,0.7312
7,8,Red Dead Redemption 2,Action,4.59,96,21,Ya,Tidak,Tidak,102.8,0.7046,0.7175
8,9,Devil May Cry 5,Action,4.25,88,9,Ya,Ya,Tidak,76.3,0.9027,0.7154
9,10,Uncharted 4: A Thief’s End,"Shooter, Action",4.48,93,0,Ya,Ya,Tidak,85.3,0.8115,0.7025


In [22]:
# ── 4.6 Demo: Rekomendasi Pemilik
print("Demo Rekomendasi Pemilik: Game yang disarankan ditambah ke stok")
owner = recommend_owner(ps_version="Both", top_k=10)
owner


Demo Rekomendasi Pemilik: Game yang disarankan ditambah ke stok


,Rank,Judul,Genre,Rating_Global,Metacritic,Waktu_Main_Jam,Bisa_PS4,Bisa_PS5,Multiplayer,Size_GB,Popularity_Score,Avg_Similarity,Skor_Akhir
0,1,The Witcher 3: Wild Hunt,"Action, RPG",4.64,92,43,Ya,Ya,Tidak,82.6,98.2,0.5588,0.7705
1,2,The Elder Scrolls V: Skyrim,"Action, RPG",4.42,94,46,Ya,Ya,Tidak,72.8,72.9,0.5738,0.6513
2,3,Fallout 4,"Action, RPG",3.81,84,38,Ya,Ya,Tidak,99.9,59.3,0.5893,0.5912
3,4,Destiny 2,"Shooter, Action",3.52,82,6,Ya,Ya,Tidak,45.5,57.7,0.5760,0.5766
4,5,God of War (2018),Action,4.54,94,15,Ya,Tidak,Tidak,71.6,66.0,0.4560,0.5578
5,6,Marvel's Spider-Man,Action,4.45,87,6,Ya,Ya,Tidak,49.9,47.6,0.6378,0.5571
6,7,Cyberpunk 2077,"Shooter, Action, RPG",4.23,73,29,Ya,Ya,Tidak,82.8,56.8,0.5428,0.5555
7,8,BioShock Infinite,"Shooter, Action",4.38,94,12,Ya,Tidak,Tidak,85.8,67.9,0.4166,0.5478
8,9,Uncharted 4: A Thief’s End,"Shooter, Action",4.48,93,0,Ya,Ya,Tidak,85.3,44.8,0.5925,0.5204
9,10,Life is Strange,Adventure,4.12,83,6,Ya,Tidak,Tidak,23.1,65.9,0.3791,0.5189


---
## 📊 Bagian 5 — Evaluasi Model

Metrik yang digunakan:
| Metrik | Deskripsi |
|--------|-----------|
| **Similarity Distribution** | Sebaran nilai cosine similarity di seluruh matrix |
| **Catalog Coverage** | % game RAWG yang pernah masuk top-K rekomendasi |
| **Genre Diversity** | Keberagaman genre dalam hasil rekomendasi |
| **Best Match per Stok** | Game RAWG paling mirip untuk setiap game rental |
| **Demo 4 Skenario** | Uji rekomendasi dengan filter berbeda |


In [23]:
# ── 5.1 Statistik Distribusi Cosine Similarity
print("=" * 55)
print("  5.1 DISTRIBUSI COSINE SIMILARITY")
print("=" * 55)
print(f"  Min    : {sim_matrix.min():.4f}")
print(f"  Max    : {sim_matrix.max():.4f}")
print(f"  Mean   : {sim_matrix.mean():.4f}")
print(f"  Median : {np.median(sim_matrix):.4f}")
print(f"  Std    : {sim_matrix.std():.4f}")
print()
for t in [0.3, 0.5, 0.7, 0.9]:
    n   = (sim_matrix > t).sum()
    pct = n / sim_matrix.size * 100
    print(f"  Pasangan > {t} : {n:>6,}  ({pct:.1f}%)")


  5.1 DISTRIBUSI COSINE SIMILARITY
  Min    : 0.0120
  Max    : 0.9995
  Mean   : 0.4285
  Median : 0.4123
  Std    : 0.1763

  Pasangan > 0.3 : 42,582  (76.1%)
  Pasangan > 0.5 : 18,433  (32.9%)
  Pasangan > 0.7 :  4,065  (7.3%)
  Pasangan > 0.9 :    211  (0.4%)


In [24]:
# ── 5.2 Catalog Coverage
print("=" * 55)
print("  5.2 CATALOG COVERAGE")
print("=" * 55)

all_rec_idx = set()
for i in range(len(df_test)):
    top_idx = np.argsort(sim_matrix[i])[::-1][:TOP_K]
    all_rec_idx.update(top_idx.tolist())

coverage = len(all_rec_idx) / len(df_train) * 100
print(f"  Game unik yang direkomendasikan : {len(all_rec_idx)}")
print(f"  Total katalog RAWG              : {len(df_train)}")
print(f"  Catalog Coverage (top-{TOP_K})    : {coverage:.2f}%")
print()
print("  ⓘ Coverage rendah (~5%) adalah normal untuk sistem top-K")
print("    karena hanya 28 game stok yang menjadi anchor query.")


  5.2 CATALOG COVERAGE
  Game unik yang direkomendasikan : 109
  Total katalog RAWG              : 1999
  Catalog Coverage (top-10)    : 5.45%

  ⓘ Coverage rendah (~5%) adalah normal untuk sistem top-K
    karena hanya 28 game stok yang menjadi anchor query.


In [25]:
# ── 5.3 Distribusi Genre dalam Rekomendasi
print("=" * 55)
print("  5.3 DISTRIBUSI GENRE REKOMENDASI (top-10 per stok)")
print("=" * 55)

genre_counter = Counter()
for i in range(len(df_test)):
    top_idx = np.argsort(sim_matrix[i])[::-1][:TOP_K]
    for idx in top_idx:
        for part in str(df_train.iloc[idx]["Genre"]).split(", "):
            genre_counter[part.strip()] += 1

df_genre = pd.DataFrame(
    genre_counter.most_common(15), columns=["Genre", "Frekuensi"]
)
df_genre["Persentase (%)"] = (df_genre["Frekuensi"] / df_genre["Frekuensi"].sum() * 100).round(1)
df_genre


  5.3 DISTRIBUSI GENRE REKOMENDASI (top-10 per stok)


,Genre,Frekuensi,Persentase (%)
0,Action,147,34.3
1,Sports,70,16.3
2,Adventure,53,12.4
3,Fighting,40,9.3
4,Racing,32,7.5
5,Simulation,25,5.8
6,Casual,24,5.6
7,Arcade,11,2.6
8,Indie,11,2.6
9,RPG,7,1.6


In [26]:
# ── 5.4 Similarity Tertinggi & Rata-rata per Game Stok
print("=" * 55)
print("  5.4 BEST MATCH & AVG SIMILARITY PER GAME STOK")
print("=" * 55)

rows = []
for i, row in df_test.iterrows():
    sim_row  = sim_matrix[i]
    best_idx = np.argmax(sim_row)
    n_above  = (sim_row >= SIMILARITY_THRESHOLD).sum()
    rows.append({
        "Game_Rental"       : row["Nama_Game"],
        "Best_Match_RAWG"   : df_train.iloc[best_idx]["Judul"],
        "Best_Similarity"   : round(float(sim_row[best_idx]), 4),
        "Avg_Similarity"    : round(float(sim_row.mean()), 4),
        f"N_≥{SIMILARITY_THRESHOLD}" : int(n_above),
    })

df_sim = pd.DataFrame(rows).sort_values("Best_Similarity", ascending=False).reset_index(drop=True)
df_sim


  5.4 BEST MATCH & AVG SIMILARITY PER GAME STOK


,Game_Rental,Best_Match_RAWG,Best_Similarity,Avg_Similarity,N_≥0.25
0,Ghost of Tsushima,Somerville,0.9995,0.6606,1999
1,Mortal Kombat XL,Ultimate Marvel vs. Capcom 3,0.9987,0.5142,1991
2,E-Football,Wo Long: Fallen Dynasty,0.9946,0.6328,1999
3,Insomniac Marvel's Spider Man 2,Forspoken,0.9900,0.5182,1793
4,NFS Heat,Hot Wheels Unleashed,0.9886,0.4440,1999
5,Tom Clancy's Ghost Recon Breakpoint,DYNASTY WARRIORS 9,0.9835,0.5930,1994
6,Red Dead Redemption 2,DYNASTY WARRIORS 9,0.9780,0.5869,1985
7,Grand Theft Auto V,Terminator: Resistance,0.9564,0.5658,1998
8,Stellar Blade,Suicide Squad: Kill The Justice League,0.9563,0.3984,1445
9,UFC 5,NBA 2K26,0.9562,0.1980,726


In [27]:
# ── 5.5 Analisis Stok Rental — Ranking Popularitas
print("=" * 55)
print("  5.5 ANALISIS STOK RENTAL — RANKING POPULARITAS")
print("=" * 55)

df_stk = df_test.copy()
df_stk = df_stk.sort_values("Total_Playtime", ascending=False).reset_index(drop=True)
df_stk.insert(0, "Rank", df_stk.index + 1)

df_stk[["Rank","Nama_Game","Genre","Total_Playtime",
        "Jumlah_Meja","Bisa_PS4","Bisa_PS5","Local_Multiplayer"]]


  5.5 ANALISIS STOK RENTAL — RANKING POPULARITAS


,Rank,Nama_Game,Genre,Total_Playtime,Jumlah_Meja,Bisa_PS4,Bisa_PS5,Local_Multiplayer
0,1,EA FC26,Sports,694,4,0,1,1
1,2,Mortal Kombat XL,Fighting,559,2,1,0,1
2,3,Tekken 8,Fighting,514,2,0,1,1
3,4,Stumble Guys,Casual,501,4,1,1,1
4,5,Ghost of Tsushima,Action-Adventure,470,2,1,1,0
5,6,Overcooked 2,Party,470,4,1,1,1
6,7,E-Football,Action,454,4,1,1,0
7,8,Black Desert,MMORPG,437,1,0,1,0
8,9,Insomniac Marvel's Spider Man 2,Action-Adventure,403,1,0,1,0
9,10,Disney Speedstorm,Casual,395,3,1,1,1


In [28]:
# ── 5.6 Demo 4 Skenario Rekomendasi Pemain
skenario = [
    {"game": "Tekken 8",           "ps": "PS5",  "multi": True,
     "genre": None,             "label": "PS5 | Multiplayer"},
    {"game": "Grand Theft Auto V",  "ps": "PS4",  "multi": False,
     "genre": None,             "label": "PS4 | Semua Genre"},
    {"game": "Overcooked 2",        "ps": "Both", "multi": True,
     "genre": ["party","casual"], "label": "Both | Multi | Party/Casual"},
    {"game": "Ghost of Tsushima",   "ps": "Both", "multi": False,
     "genre": ["action"],       "label": "Both | Action"},
]

for s in skenario:
    print("=" * 55)
    print(f"  Game: '{s['game']}'  [{s['label']}]")
    print("=" * 55)
    try:
        reko = recommend_player(
            s["game"], ps_version=s["ps"],
            multiplayer_only=s["multi"], genre_filter=s["genre"]
        )
        display(reko[["Rank","Judul","Genre","Rating_Global","Similarity","Skor_Akhir"]])
    except ValueError as e:
        print(f"  [!] {e}")
    print()


  Game: 'Tekken 8'  [PS5 | Multiplayer]


,Rank,Judul,Genre,Rating_Global,Similarity,Skor_Akhir
0,1,Mortal Kombat 1,"Action, Fighting",3.60,0.9045,0.6398
1,2,Tekken 8,"Action, Fighting",4.03,0.9037,0.6382
2,3,Guilty Gear -Strive,"Action, Fighting",4.24,0.8433,0.6016
3,4,Nickelodeon All-Star Brawl,"Action, Fighting",2.86,0.8077,0.5787
4,5,Street Fighter 6,"Adventure, Action, Fighting",4.11,0.7818,0.5538
5,6,Them's Fightin' Herds,"Indie, Action, Fighting",3.36,0.7728,0.5503
6,7,Baldur's Gate III,"Strategy, Adventure, RPG",4.44,0.6749,0.5306
7,8,Sea of Stars,RPG,4.28,0.7198,0.5145
8,9,It Takes Two,"Platformer, Adventure, Action",4.46,0.6218,0.5097
9,10,Split Fiction,"Adventure, Action",4.45,0.7090,0.5072



  Game: 'Grand Theft Auto V'  [PS4 | Semua Genre]


,Rank,Judul,Genre,Rating_Global,Similarity,Skor_Akhir
0,1,Grand Theft Auto V,Action,4.47,0.8083,0.8658
1,2,The Witcher 3: Wild Hunt,"Action, RPG",4.64,0.7634,0.8290
2,3,The Elder Scrolls V: Skyrim,"Action, RPG",4.42,0.7823,0.7662
3,4,Fallout 4,"Action, RPG",3.81,0.8254,0.7556
4,5,Marvel's Spider-Man,Action,4.45,0.8680,0.7506
5,6,Dead by Daylight,Action,3.50,0.9456,0.7328
6,7,Destiny 2,"Shooter, Action",3.52,0.7971,0.7312
7,8,Red Dead Redemption 2,Action,4.59,0.7046,0.7175
8,9,Devil May Cry 5,Action,4.25,0.9027,0.7154
9,10,Uncharted 4: A Thief’s End,"Shooter, Action",4.48,0.8115,0.7025



  Game: 'Overcooked 2'  [Both | Multi | Party/Casual]


,Rank,Judul,Genre,Rating_Global,Similarity,Skor_Akhir
0,1,Among Us,"Casual, Action, Simulation",3.83,0.7565,0.6233
1,2,The Jackbox Party Pack 5,Casual,4.13,0.7476,0.5343
2,3,The Jackbox Party Pack 6,Casual,4.19,0.7458,0.5321
3,4,UNO,"Casual, Family",3.72,0.6580,0.4874
4,5,Overcooked! 2,"Casual, Indie, Action",4.15,0.6169,0.4858
5,6,Rocksmith 2014 Edition - Remastered,"Casual, Simulation",4.38,0.6740,0.4852
6,7,Teenage Mutant Ninja Turtles: Shredder's Revenge,"Adventure, Action, Fighting, Casual, Arcade, I...",4.08,0.6418,0.4818
7,8,MONOPOLY PLUS,"Casual, Family",3.49,0.6667,0.4809
8,9,The Jackbox Party Pack 7,"Casual, Indie",4.30,0.6747,0.4802
9,10,Brawlhalla,"Casual, Indie, Action, Fighting",3.22,0.5467,0.4774



  Game: 'Ghost of Tsushima'  [Both | Action]


,Rank,Judul,Genre,Rating_Global,Similarity,Skor_Akhir
0,1,Grand Theft Auto V,Action,4.47,0.8028,0.8620
1,2,The Witcher 3: Wild Hunt,"Action, RPG",4.64,0.7483,0.8184
2,3,Star Wars Jedi: Fallen Order,"Adventure, Action",4.13,0.9816,0.8074
3,4,A Plague Tale: Innocence,"Adventure, Action",4.15,0.9848,0.7770
4,5,Resident Evil 2,"Adventure, Action",4.50,0.9808,0.7657
5,6,The Elder Scrolls V: Skyrim,"Action, RPG",4.42,0.7716,0.7587
6,7,Resident Evil: Village,"Adventure, Action",4.39,0.9874,0.7538
7,8,Control,"Shooter, Adventure, Action",4.14,0.9027,0.7505
8,9,Resident Evil 3,"Adventure, Action",4.04,0.9785,0.7504
9,10,Uncharted: The Lost Legacy,"Adventure, Action",4.27,0.9906,0.7487


In [29]:
# ── 5.7 Rekomendasi Pemilik — Game Baru untuk Ditambah ke Stok
print("=" * 55)
print("  5.7 REKOMENDASI PEMILIK — TOP-10 GAME BARU")
print("=" * 55)

owner_full = recommend_owner(ps_version="Both", top_k=10)
owner_full


  5.7 REKOMENDASI PEMILIK — TOP-10 GAME BARU


,Rank,Judul,Genre,Rating_Global,Metacritic,Waktu_Main_Jam,Bisa_PS4,Bisa_PS5,Multiplayer,Size_GB,Popularity_Score,Avg_Similarity,Skor_Akhir
0,1,The Witcher 3: Wild Hunt,"Action, RPG",4.64,92,43,Ya,Ya,Tidak,82.6,98.2,0.5588,0.7705
1,2,The Elder Scrolls V: Skyrim,"Action, RPG",4.42,94,46,Ya,Ya,Tidak,72.8,72.9,0.5738,0.6513
2,3,Fallout 4,"Action, RPG",3.81,84,38,Ya,Ya,Tidak,99.9,59.3,0.5893,0.5912
3,4,Destiny 2,"Shooter, Action",3.52,82,6,Ya,Ya,Tidak,45.5,57.7,0.5760,0.5766
4,5,God of War (2018),Action,4.54,94,15,Ya,Tidak,Tidak,71.6,66.0,0.4560,0.5578
5,6,Marvel's Spider-Man,Action,4.45,87,6,Ya,Ya,Tidak,49.9,47.6,0.6378,0.5571
6,7,Cyberpunk 2077,"Shooter, Action, RPG",4.23,73,29,Ya,Ya,Tidak,82.8,56.8,0.5428,0.5555
7,8,BioShock Infinite,"Shooter, Action",4.38,94,12,Ya,Tidak,Tidak,85.8,67.9,0.4166,0.5478
8,9,Uncharted 4: A Thief’s End,"Shooter, Action",4.48,93,0,Ya,Ya,Tidak,85.3,44.8,0.5925,0.5204
9,10,Life is Strange,Adventure,4.12,83,6,Ya,Tidak,Tidak,23.1,65.9,0.3791,0.5189


In [30]:
# ── 5.8 Ringkasan Metrik Evaluasi
print("=" * 55)
print("  5.8 RINGKASAN METRIK EVALUASI")
print("=" * 55)

summary = {
    "Total game katalog RAWG (train)" : len(df_train),
    "Total game stok rental (test)"   : len(df_test),
    "Total fitur model"               : len(feature_cols),
    "Ukuran similarity matrix"        : f"{sim_matrix.shape[0]} × {sim_matrix.shape[1]}",
    "Similarity threshold"            : SIMILARITY_THRESHOLD,
    f"Catalog Coverage (top-{TOP_K})"  : f"{coverage:.2f}%",
    "Avg Similarity (semua pasangan)" : f"{sim_matrix.mean():.4f}",
    "Max Similarity"                  : f"{sim_matrix.max():.4f}",
    "Min Similarity"                  : f"{sim_matrix.min():.4f}",
    "Pasangan ≥ threshold"            : f"{(sim_matrix >= SIMILARITY_THRESHOLD).sum():,}",
    "Jumlah genre OHE"                : len(genre_cols),
}

df_summary = pd.DataFrame(
    list(summary.items()), columns=["Metrik", "Nilai"]
)
df_summary


  5.8 RINGKASAN METRIK EVALUASI


,Metrik,Nilai
0,Total game katalog RAWG (train),1999
1,Total game stok rental (test),28
2,Total fitur model,27
3,Ukuran similarity matrix,28 × 1999
4,Similarity threshold,0.25
5,Catalog Coverage (top-10),5.45%
6,Avg Similarity (semua pasangan),0.4285
7,Max Similarity,0.9995
8,Min Similarity,0.0120
9,Pasangan ≥ threshold,"48,937"


---
## ✅ Selesai

Model Item-Based Collaborative Filtering untuk sistem rekomendasi game rental PS telah berhasil dibangun dan dievaluasi.

**Cara menggunakan fungsi rekomendasi:**

```python
# Rekomendasi untuk pemain
recommend_player(
    game_name        = "Tekken 8",   # game yang dipilih dari stok
    ps_version       = "PS5",        # "PS4" | "PS5" | "Both"
    multiplayer_only = True,         # True jika hanya ingin game multiplayer
    genre_filter     = ["action"],   # filter genre (opsional)
    meja             = 1,            # filter meja (opsional)
    max_size_gb      = 80,           # batas ukuran game (opsional)
    top_k            = 10,
)

# Rekomendasi game baru untuk pemilik rental
recommend_owner(
    ps_version       = "Both",
    multiplayer_only = False,
    top_k            = 10,
)
```


---
## 💾 Bagian 6 — Simpan Model ke Pickle

Menyimpan model yang sudah dilatih (similarity matrix, dataframes, scaler, feature columns) ke file `.pkl` untuk digunakan di aplikasi lain.


In [31]:
# ── 6.1 Siapkan Model Package
import pickle

# Kumpulkan semua komponen model dalam satu dictionary
model_package = {
    # ── Data training (sudah dipreproses)
    "df_train": df_train,
    "df_test": df_test,
    
    # ── Feature information
    "feature_cols": feature_cols,
    "numeric_features": NUMERIC_FEATURES,
    "genre_cols": genre_cols,
    
    # ── Similarity matrix
    "sim_matrix": sim_matrix,
    
    # ── Scaler
    "scaler": scaler,
    "scaler_knn": scaler_knn if 'scaler_knn' in locals() else None,
    
    # ── Hyperparameters
    "top_k": TOP_K,
    "similarity_threshold": SIMILARITY_THRESHOLD,
    "col_proxy": COL_PROXY,
    
    # ── Model configuration
    "model_version": "1.0",
    "model_date": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
}

print("✅ Model package siap untuk disimpan")
print(f"   Komponen: {list(model_package.keys())}")


✅ Model package siap untuk disimpan
   Komponen: ['df_train', 'df_test', 'feature_cols', 'numeric_features', 'genre_cols', 'sim_matrix', 'scaler', 'scaler_knn', 'top_k', 'similarity_threshold', 'col_proxy', 'model_version', 'model_date']
